# Predictive Modelling

A fitted model is useful when it predicts new observations well. These notes develop the prediction target, explain why training error is optimistic, and compare two ways to estimate prediction error: complexity penalties and held-out predictions.

:::{admonition} The prediction target
:class: important

Under squared loss, predict the conditional mean: $m(x)=E[Y\mid X=x]$. The challenge is estimating it from finite data.
:::

## Topics

::::{grid} 1 1 2 2

:::{grid-item-card} Predictive Modelling

What should we predict, and how can we estimate it?
:::

:::{grid-item-card} [Prediction Error & Bias-Variance](Prediction-Error.ipynb)

Which error do we want to estimate?
:::

:::{grid-item-card} [Optimism & Model Complexity](Optimism-and-Complexity.ipynb)

How much should we penalize training error?
:::

:::{grid-item-card} [Cross-Validation](Cross-Validation.ipynb)

How can held-out data guide model selection?
:::

:::{grid-item-card} [LOO & Generalized Cross-Validation](LOO-and-GCV.ipynb)

When can we estimate error from a single fit?
:::
::::

## Inputs, Outputs & Learning Algorithms

Let the training set be

$$T=\{(X_1,Y_1),\ldots,(X_n,Y_n)\},\qquad (X_i,Y_i)\overset{\mathrm{iid}}{\sim}P_{XY}.$$

For regression, $X_i\in\mathbb R^d$ and $Y_i\in\mathbb R$. A new pair $(X_0,Y_0)$ is independent of $T$ and comes from the same distribution.

| Object | Notation | Meaning |
|---|---|---|
| Learning algorithm | $A$ | A procedure that takes training data and returns a function. |
| Fitted prediction rule | $\widehat f_T=A(T)$ | The particular function learned from this training set. |
| New prediction | $\widehat f_T(x_0)$ | A numerical prediction at a new input. |
| Regression function | $m(x)=E[Y\mid X=x]$ | The population conditional mean; generally unknown. |
| Tuning parameter | $\lambda$, $K$, or polynomial degree $r$ | A choice controlling the learning procedure. |

For example, least squares is an algorithm; $\widehat f_T(x)=2+3x$ is one fitted rule. A different training set generally produces different coefficients.

**Notation:** $d$ counts input variables; $p$ counts fitted regression coefficients, including an intercept when present. $K$ denotes neighbours and $k$ denotes CV folds.

## Loss Functions

A **loss function** $L(y,a)$ measures the cost of predicting $a$ when the outcome is $y$. The choice of loss defines what a good prediction means.

| Loss | Formula | Population-optimal prediction at $x$ |
|---|---|---|
| Squared error | $(y-a)^2$ | Conditional mean $E[Y\mid X=x]$. |
| Absolute error | $\lvert y-a\rvert$ | A conditional median. |
| Zero-one loss, for class labels | $I(y\ne a)$ | A most probable class given $x$. |

These notes primarily use **squared error**. It penalizes large errors strongly and permits useful decompositions.

:::{admonition} Why Squared Error Appears in Gaussian Likelihoods
:class: note

If independent responses satisfy $Y_i\mid X_i\sim N(f(X_i),\sigma^2)$ with common fixed $\sigma^2$, then

$$-\ell(f)=\frac n2\log(2\pi\sigma^2)+\frac{1}{2\sigma^2}\sum_{i=1}^n[Y_i-f(X_i)]^2.$$

The first term does not depend on $f$, so maximizing the likelihood over a specified model class is equivalent to minimizing the sum of squared errors. This connection does **not** mean Gaussian errors are necessary for the conditional mean to minimize squared loss.
:::

:::{admonition} A Local Taylor Motivation
:class: note

Suppose loss depends only on the prediction error: $L(y,a)=\ell(y-a)$. If $\ell$ is twice continuously differentiable near zero, minimized at zero, and $\ell(0)=0$, then $\ell'(0)=0$ and

$$\ell(h)=\tfrac12\ell''(0)h^2+o(h^2).$$

When $\ell''(0)>0$, squared error is the leading local approximation. This is a **local** result, not an equivalence for large errors or nonsmooth losses such as absolute error. A minimum alone gives $\ell''(0)\geq0$; if it is zero, a higher-order term may be needed.
:::

## Why the Conditional Mean Is Optimal

For any deterministic prediction $a$ at $x$, assuming finite second moments,

$$\boxed{E[(Y-a)^2\mid X=x]=\operatorname{Var}(Y\mid X=x)+[m(x)-a]^2.}$$

The first term does not depend on $a$. The second is minimized at $a=m(x)$, giving the **regression function** as the best squared-error predictor.

Over all measurable prediction functions,

$$\boxed{\arg\min_f E[(Y-f(X))^2]=m,\qquad m(x)=E[Y\mid X=x],}$$

with equality understood up to inputs of probability zero. The minimum possible risk is $E_X[\operatorname{Var}(Y\mid X)]$.

:::{dropdown} Derivation: Add and Subtract the Conditional Mean
Write

$$Y-a=[Y-m(x)]+[m(x)-a].$$

Expanding and conditioning on $X=x$ gives

$$\begin{aligned}
E[(Y-a)^2\mid X=x]
&=E[(Y-m(x))^2\mid X=x]+[m(x)-a]^2\\
&\quad+2[m(x)-a]E[Y-m(x)\mid X=x].
\end{aligned}$$

The final expectation is zero by the definition of $m(x)$. The remaining two terms are conditional variance and squared distance from the optimal prediction. Averaging over $X$ gives

$$E[(Y-f(X))^2]=E[\operatorname{Var}(Y\mid X)]+E[(m(X)-f(X))^2].$$
:::

:::{admonition} Worked Example: Mean vs. Individual Outcome
:class: tip callout-example

At a fixed $x$, suppose $Y=2$ with probability $1/4$ and $Y=6$ with probability $3/4$.

$$m(x)=5,\qquad \operatorname{Var}(Y\mid X=x)=3.$$

Predicting $5$ has expected squared loss $3$. Predicting $6$ has expected squared loss $4=3+(5-6)^2$. The best squared-error prediction need not be a possible observed outcome.
:::

## Estimating the Regression Function

### Exact-Input Averages

A direct estimate averages responses at a repeated input:

$$\widehat m(x)=\frac{\sum_{i:X_i=x}Y_i}{\#\{i:X_i=x\}}.$$

It becomes precise with many independent replicates at that input. With continuous inputs, exact repeats are rare. At an unobserved input the denominator is zero, so this rule cannot even interpolate between observed values.

### Two Ways to Share Information

::::::{tab-set}

:::::{tab-item} Local: nearest neighbours

Let $N_K(x)$ contain the indices of the $K$ nearest training inputs to $x$, with a fixed tie rule. Then

$$\widehat f_K(x)=\frac1K\sum_{i\in N_K(x)}Y_i.$$

- **Small $K$:** more local and flexible, but more sensitive to individual observations.
- **Large $K$:** more averaging, usually lower variance but more smoothing bias.
- **Distance matters:** variables measured on larger scales can dominate Euclidean distance. Estimate any scaling from the training portion of each split.

For homoscedastic independent errors and a fixed neighbourhood, the conditional noise variance of this average is $\sigma^2/K$. Its bias depends on how the neighbours' means differ from $m(x)$. Nearest-neighbour predictions are typically piecewise constant, so “smoothing” does not imply mathematical continuity.

:::{admonition} The Curse of Dimensionality
:class: note

For uniformly distributed inputs in a $d$-dimensional unit cube, a subcube with side length $h$ occupies fraction $h^d$ of the volume. To contain about $K$ of $n$ observations, its side length must be roughly

$$h\approx(K/n)^{1/d}.$$

If $K/n=0.01$, then $h=0.01$ in one dimension but $h\approx0.631$ in ten dimensions. A neighbourhood holding only 1% of the data may span much of each coordinate range. Keeping $h$ fixed requires $n\approx K/h^d$, exponential in dimension in this example.
:::
:::::

:::::{tab-item} Global: polynomial models

For one input, approximate the regression function by

$$f_r(x)=\beta_0+\beta_1(x-\bar x)+\cdots+\beta_r(x-\bar x)^r.$$

Fit the coefficients using all observations. A Taylor expansion motivates the basis for a sufficiently smooth function, but fitted coefficients are estimated from data rather than assumed to be known derivatives. This model is **linear in its coefficients**, even when it is curved in $x$.

| Approach | Strength | Limitation |
|---|---|---|
| Local averaging | Adapts without specifying a single global polynomial shape. | Needs enough nearby data; distances become less informative in high dimension. |
| Global parametric model | Shares information across all observations; can be precise when its assumptions fit. | Misspecification can cause systematic error, especially outside the observed range. |

Both approaches require a choice of complexity. Training error alone cannot make that choice reliably.
:::::
::::::